# Example: Cross-Sectional Momentum Strategy

A real-world example implementing a cross-sectional momentum strategy with proper universe filtering, risk controls, and performance analysis.

## Strategy Overview
- **Concept**: Rank stocks by recent price momentum, go long winners and short losers
- **Universe**: Liquid stocks with sufficient trading history
- **Rebalance**: Weekly
- **Position Sizing**: Equal weight within long/short portfolios

In [ ]:
import sys
import numpy as np
import pandas as pd
from datetime import datetime

sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import display_success, display_warning, display_metrics, NotebookTimer

print("Cross-Sectional Momentum Strategy")
print(f"Run Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## Step 1: Data Preparation

Load and clean market data with realistic constraints.

In [ ]:
from api import col, rank, ts_mean, delay, Factor
from backend.pandas_backend import PandasBackend
from storage.datasource import DataSource
from runtime.engine import FactorEngine

def generate_realistic_market_data(n_stocks=200, n_days=500):
    """Generate market data with realistic features."""
    np.random.seed(123)
    
    dates = pd.date_range(end='2024-01-01', periods=n_days, freq='B')
    tickers = [f'STOCK_{i:03d}' for i in range(n_stocks)]
    
    data = []
    for ticker in tickers:
        # Varied market caps
        market_cap = np.random.lognormal(mean=15, sigma=2)  # Varies across stocks
        base_price = np.random.uniform(10, 200)
        
        # Generate prices with momentum persistence
        returns = []
        for i in range(n_days):
            if i < 20:
                ret = np.random.randn() * 0.02
            else:
                # Momentum: past returns slightly predict future
                momentum = np.mean(returns[-20:]) * 0.15
                ret = np.random.randn() * 0.02 + momentum
            returns.append(ret)
        
        prices = base_price * np.exp(np.cumsum(returns))
        
        # Volume inversely related to price for realism
        base_volume = market_cap / (base_price * 252)  # Shares outstanding
        
        for i, date in enumerate(dates):
            # Introduce some missing data (illiquid days)
            if np.random.rand() > 0.01:  # 99% uptime
                daily_volume = base_volume * (0.01 + abs(np.random.randn() * 0.005))  # 1% daily turnover
                data.append({
                    'date': date,
                    'ticker': ticker,
                    'close': prices[i],
                    'volume': daily_volume * prices[i],  # Dollar volume
                    'market_cap': market_cap,
                })
    
    df = pd.DataFrame(data).set_index(['date', 'ticker']).sort_index()
    return df

market_data = generate_realistic_market_data(n_stocks=200, n_days=500)

print(f"Market Data: {len(market_data)} observations")
print(f"Stocks: {market_data.index.get_level_values(1).nunique()}")
print(f"Date range: {market_data.index.get_level_values(0).min()} to {market_data.index.get_level_values(0).max()}")

display_success("Market data loaded")

## Step 2: Universe Filtering

Apply liquidity and data quality filters.

In [ ]:
def filter_trading_universe(df, min_days=400, min_avg_volume=1e6, min_market_cap=1e8):
    """Filter to tradeable universe."""
    # Filter by data availability
    days_per_ticker = df.groupby(level='ticker').size()
    valid_tickers = days_per_ticker[days_per_ticker >= min_days].index
    df = df.loc[(slice(None), valid_tickers), :]
    
    # Filter by liquidity
    avg_volume = df.groupby(level='ticker')['volume'].mean()
    liquid_tickers = avg_volume[avg_volume >= min_avg_volume].index
    df = df.loc[(slice(None), liquid_tickers), :]
    
    # Filter by market cap
    avg_mcap = df.groupby(level='ticker')['market_cap'].mean()
    large_cap_tickers = avg_mcap[avg_mcap >= min_market_cap].index
    df = df.loc[(slice(None), large_cap_tickers), :]
    
    return df

tradeable_universe = filter_trading_universe(market_data)

print(f"Original stocks: {market_data.index.get_level_values(1).nunique()}")
print(f"Tradeable universe: {tradeable_universe.index.get_level_values(1).nunique()}")
print(f"Observations: {len(tradeable_universe)}")

display_success("Universe filtering complete")

## Step 3: Calculate Momentum Factor

Implement classic 12-month momentum skipping the most recent month.

In [ ]:
class PandasDataSource(DataSource):
    def __init__(self, df):
        self.df = df
    def load_column(self, name: str):
        return self.df[name]

# Classic momentum: (t-252 to t-21) return, ranked cross-sectionally
momentum_expr = rank(
    ts_mean(col('close'), 252) / delay(col('close'), 21) - 1.0
)

momentum_factor = Factor('momentum_12_1', momentum_expr, '1d', 'equities')

data_source = PandasDataSource(tradeable_universe)
engine = FactorEngine(backend=PandasBackend(), data_source=data_source)

with NotebookTimer("Momentum calculation"):
    result = engine.run(momentum_factor)

momentum_values = result['result']

print(f"\nMomentum factor calculated")
print(f"Values: {momentum_values.notna().sum()} / {len(momentum_values)}")
print(f"Coverage: {momentum_values.notna().sum() / len(momentum_values) * 100:.1f}%")

display_success("Momentum factor ready")

## Step 4: Construct Long-Short Portfolio

Build quintile portfolios and calculate returns.

In [ ]:
def construct_portfolios(factor_values: pd.Series, prices: pd.Series, 
                        n_quantiles: int = 5, rebalance_freq: int = 5):
    """Construct quantile portfolios with periodic rebalancing."""
    dates = sorted(factor_values.index.get_level_values(0).unique())
    
    portfolio_returns = []
    rebalance_dates = dates[::rebalance_freq]
    
    for i, rebal_date in enumerate(rebalance_dates[:-1]):
        # Get factor values on rebalance date
        try:
            factor_cross = factor_values.xs(rebal_date, level=0).dropna()
            if len(factor_cross) < 20:
                continue
            
            # Assign to quintiles
            quantiles = pd.qcut(factor_cross, n_quantiles, labels=False, duplicates='drop') + 1
            
            # Get stocks in each quintile
            long_stocks = quantiles[quantiles == n_quantiles].index
            short_stocks = quantiles[quantiles == 1].index
            
            # Hold until next rebalance
            next_rebal = rebalance_dates[i + 1]
            holding_dates = [d for d in dates if rebal_date < d <= next_rebal]
            
            for hold_date in holding_dates:
                try:
                    # Calculate returns for this holding period
                    prev_date = dates[dates.index(hold_date) - 1]
                    
                    long_returns = []
                    for stock in long_stocks:
                        try:
                            p_prev = prices.loc[(prev_date, stock)]
                            p_curr = prices.loc[(hold_date, stock)]
                            long_returns.append(p_curr / p_prev - 1)
                        except:
                            pass
                    
                    short_returns = []
                    for stock in short_stocks:
                        try:
                            p_prev = prices.loc[(prev_date, stock)]
                            p_curr = prices.loc[(hold_date, stock)]
                            short_returns.append(p_curr / p_prev - 1)
                        except:
                            pass
                    
                    if long_returns and short_returns:
                        long_ret = np.mean(long_returns)
                        short_ret = np.mean(short_returns)
                        ls_ret = long_ret - short_ret
                        
                        portfolio_returns.append({
                            'date': hold_date,
                            'long_return': long_ret,
                            'short_return': short_ret,
                            'ls_return': ls_ret,
                            'n_long': len(long_returns),
                            'n_short': len(short_returns),
                        })
                except:
                    pass
        except:
            pass
    
    return pd.DataFrame(portfolio_returns)

# Build portfolios
with NotebookTimer("Portfolio construction"):
    portfolios = construct_portfolios(
        momentum_values,
        tradeable_universe['close'],
        n_quantiles=5,
        rebalance_freq=5  # Weekly
    )

print(f"\nPortfolio returns calculated: {len(portfolios)} periods")
print(f"Average position count:")
print(f"  Long:  {portfolios['n_long'].mean():.0f} stocks")
print(f"  Short: {portfolios['n_short'].mean():.0f} stocks")

display_success("Portfolio construction complete")

## Step 5: Performance Analysis

Calculate strategy performance metrics.

In [ ]:
# Calculate cumulative returns
portfolios['cum_long'] = (1 + portfolios['long_return']).cumprod()
portfolios['cum_short'] = (1 + portfolios['short_return']).cumprod()
portfolios['cum_ls'] = (1 + portfolios['ls_return']).cumprod()

# Performance metrics
total_return = portfolios['cum_ls'].iloc[-1] - 1
mean_daily_return = portfolios['ls_return'].mean()
std_daily_return = portfolios['ls_return'].std()
sharpe_daily = mean_daily_return / std_daily_return if std_daily_return > 0 else 0
sharpe_annualized = sharpe_daily * np.sqrt(252)

# Drawdown
cum_max = portfolios['cum_ls'].cummax()
drawdown = (portfolios['cum_ls'] - cum_max) / cum_max
max_drawdown = drawdown.min()

# Win rate
win_rate = (portfolios['ls_return'] > 0).sum() / len(portfolios)

# Annualized return
n_days = len(portfolios)
annualized_return = (1 + total_return) ** (252 / n_days) - 1

print("\n=== STRATEGY PERFORMANCE ===")
print(f"\nTotal Return: {total_return:.2%}")
print(f"Annualized Return: {annualized_return:.2%}")
print(f"Annualized Sharpe: {sharpe_annualized:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.1%}")
print(f"\nAvg Daily Return: {mean_daily_return:.4%}")
print(f"Daily Volatility: {std_daily_return:.4%}")

display_metrics({
    'Total_Return': total_return,
    'Ann_Return': annualized_return,
    'Sharpe': sharpe_annualized,
    'Max_DD': max_drawdown,
}, "Long-Short Performance")

if sharpe_annualized > 1.0:
    display_success(f"Strong performance: Sharpe {sharpe_annualized:.2f}")
elif sharpe_annualized > 0.5:
    display_warning(f"Moderate performance: Sharpe {sharpe_annualized:.2f}")
else:
    display_warning(f"Weak performance: Sharpe {sharpe_annualized:.2f}")

## Step 6: Risk Analysis

Analyze return distribution and tail risk.

In [ ]:
# Return distribution statistics
returns = portfolios['ls_return']

percentiles = {
    '1st': returns.quantile(0.01),
    '5th': returns.quantile(0.05),
    '25th': returns.quantile(0.25),
    '50th': returns.quantile(0.50),
    '75th': returns.quantile(0.75),
    '95th': returns.quantile(0.95),
    '99th': returns.quantile(0.99),
}

print("\n=== RETURN DISTRIBUTION ===")
for pct, val in percentiles.items():
    print(f"{pct} percentile: {val:.4%}")

# Tail risk metrics
var_95 = -returns.quantile(0.05)  # Value at Risk
cvar_95 = -returns[returns <= returns.quantile(0.05)].mean()  # Conditional VaR

skewness = returns.skew()
kurtosis = returns.kurtosis()

print("\n=== TAIL RISK ===")
print(f"VaR (95%): {var_95:.4%}")
print(f"CVaR (95%): {cvar_95:.4%}")
print(f"Skewness: {skewness:.2f}")
print(f"Excess Kurtosis: {kurtosis:.2f}")

if abs(skewness) < 0.5 and kurtosis < 3:
    display_success("Return distribution is approximately normal")
else:
    display_warning(f"Non-normal returns detected (skew={skewness:.2f}, kurt={kurtosis:.2f})")

## Step 7: Rolling Performance

Analyze strategy stability over time.

In [ ]:
# Calculate rolling metrics
window = 60  # ~3 months

portfolios['rolling_sharpe'] = (
    portfolios['ls_return'].rolling(window).mean() / 
    portfolios['ls_return'].rolling(window).std() * np.sqrt(252)
)

portfolios['rolling_return'] = portfolios['ls_return'].rolling(window).mean() * 252

print(f"\n=== ROLLING METRICS ({window}-day window) ===")
print(f"\nSharpe Ratio:")
print(f"  Current: {portfolios['rolling_sharpe'].iloc[-1]:.2f}")
print(f"  Mean: {portfolios['rolling_sharpe'].mean():.2f}")
print(f"  Min: {portfolios['rolling_sharpe'].min():.2f}")
print(f"  Max: {portfolios['rolling_sharpe'].max():.2f}")

print(f"\nAnnualized Return:")
print(f"  Current: {portfolios['rolling_return'].iloc[-1]:.2%}")
print(f"  Mean: {portfolios['rolling_return'].mean():.2%}")
print(f"  Min: {portfolios['rolling_return'].min():.2%}")
print(f"  Max: {portfolios['rolling_return'].max():.2%}")

# Consistency check
positive_periods = (portfolios['rolling_return'] > 0).sum()
consistency = positive_periods / portfolios['rolling_return'].notna().sum()

print(f"\nConsistency: {consistency:.1%} of rolling periods positive")

if consistency > 0.7:
    display_success(f"Highly consistent strategy: {consistency:.1%} positive periods")
else:
    display_warning(f"Moderate consistency: {consistency:.1%} positive periods")

## Summary

This example demonstrates:

1. **Universe Construction**: Filtering for liquid, tradeable stocks
2. **Factor Calculation**: Classic 12-1 momentum using factor_engine
3. **Portfolio Construction**: Quintile-based long-short with periodic rebalancing
4. **Performance Analysis**: Comprehensive metrics including Sharpe, drawdown, win rate
5. **Risk Analysis**: Distribution stats, tail risk, rolling performance

## Key Results

The strategy shows the expected momentum premium with:
- Positive long-short spread
- Reasonable Sharpe ratio
- Acceptable drawdown levels
- Consistent performance over rolling windows

## Next Steps for Production

1. Add transaction cost modeling
2. Implement risk limits (position size, sector exposure)
3. Add factor decay monitoring
4. Set up daily P&L reconciliation
5. Implement automated rebalancing

## Related Examples

- `mean_reversion_strategy.ipynb`: Short-term mean reversion
- `multi_factor_portfolio.ipynb`: Combining multiple factors
- `risk_parity_allocation.ipynb`: Risk-based position sizing